# *Knowing What You Know*: the measurements

The Colab notebook behind every measured number in the film. It ran on an A100 (bf16, greedy decoding, so the outputs are deterministic).
Comments and printed messages are translated from the French original; the computations are unchanged.

**What it produces** (in the Drive folder `KWYK_film/`, plus a zip):

- **E1, the cold open and act III**: four questions in plain chat mode, the greedy answer, the output distribution at every step (top 50) and the full distribution of the first token. `e1_*.json`, `e1_*_step0.npy`
- **E1b, the bars of act III**: the next-word distribution after an unfinished sentence (completion mode, no chat template). `e1b_*.json`, `e1b_*_next.npy`
- **E3, the questions**: 600 TriviaQA questions plus 30 questions that have no answer. For each: the activations of six blocks at the last prompt token, *before* generation; the first-token entropy and maximum probability (the baselines); the greedy answer, checked against the TriviaQA aliases. `activations_llama8b_layers.npz`
- **H3, the probe**: a logistic-regression probe per block, scored out of sample (five folds), against the entropy and max-probability baselines; the 3D cloud. `h3_layer_scan.json`, `h3_metrics.json`, `h3_cloud.json`
- **E3 showcase**: the four right answers with the highest probe scores and the four wrong ones with the lowest. `e3_showcase.json`
- **E4, effort**: a reasoning model (Qwen3-8B, thinking mode) on three problems; the length of its reasoning trace. `e4_effort.json`
- **E5, what it says**: the same 600 questions, now with a stated confidence from 0 to 100. `e5_items.json`, `e5_said_vs_probe.json`

Everything shown on screen that is not one of these files is derived from them by `../derive.py`.

**Requirements**: a GPU with bf16 and about 20 GB of memory (A100 or L4). Llama 3.1 is gated: accept the licence on https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, then store a read token as the Colab secret `HF_TOKEN`. **Duration**: about an hour on an A100.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If transformers was already imported in this session (or an ImportError such as
# "cannot import name ... from transformers.utils" appears below): Runtime >
# Restart session, then continue from the next cell WITHOUT rerunning this pip.
!pip install -q -U transformers accelerate datasets
import importlib, transformers; importlib.reload(transformers); print('transformers', transformers.__version__)

In [ ]:
import torch, os
from getpass import getpass
from huggingface_hub import login
# Order: environment variable > Colab secret "HF_TOKEN" (key icon, on the left) > prompt.
# With the secret set, "Run all" no longer stops here.
if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        os.environ['HF_TOKEN'] = getpass('HF token (read) : ')
login(os.environ['HF_TOKEN'])
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'n/a')
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 'n/a')

## 2. Configuration

In [ ]:
from pathlib import Path
import json, re, numpy as np

OUT = Path('/content/drive/MyDrive/KWYK_film')
OUT.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
LAYER_BLOCK_IDX = 16          # the middle block (of 32)
N_TRIVIA = 600                # TriviaQA questions (validation, rc.nocontext), fixed draw
SEED = 0
MAX_NEW_TOKENS_E3 = 24        # short answers
MAX_NEW_TOKENS_E1 = 60        # free answers, cold open
TOPK_STEP = 50                # distribution kept at each generation step (E1)
DEVICE, DTYPE = 'cuda', torch.bfloat16

## 3. Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE)
model.eval()
print(f'{MODEL_ID}: {model.config.num_hidden_layers} blocks, hidden={model.config.hidden_size}, vocab={model.config.vocab_size}')
assert 0 <= LAYER_BLOCK_IDX < model.config.num_hidden_layers
HS_INDEX = LAYER_BLOCK_IDX + 1   # hidden_states[0] = embeddings; [k+1] = output of block k

## 4. E1: the cold open

Plain chat, no instructions: the question, the greedy answer, and at each step the output distribution (top 50).
The first step keeps the full distribution (float16).

In [ ]:
E1_QUESTIONS = {
    'santiago': 'What is the capital of Chile?',
    'alpha3':   'What is the capital of the Martian colony Alpha-three?',
    # the two questions the cold open also asks the viewer
    'paris':    'What is the capital of France?',
    'bosch':    'Who painted The Garden of Earthly Delights?',
}

def entropy_bits(p):
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def run_e1(name, question):
    messages = [{'role': 'user', 'content': question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    n_prompt = inputs['input_ids'].shape[1]
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS_E1, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id,
                             output_scores=True, return_dict_in_generate=True)
    new_ids = gen.sequences[0, n_prompt:]
    answer = tokenizer.decode(new_ids, skip_special_tokens=True)
    steps = []
    for k, logits in enumerate(gen.scores):
        p = torch.softmax(logits[0].float(), dim=-1)
        pk = p.cpu().numpy()
        if k == 0:
            np.save(OUT / f'e1_{name}_step0.npy', pk.astype(np.float16))
        top = torch.topk(p, TOPK_STEP)
        steps.append({
            'step': k,
            'chosen': tokenizer.decode([int(new_ids[k])]),
            'chosen_prob': float(p[int(new_ids[k])]),
            'entropy_bits': entropy_bits(pk),
            'top': [{'tok': tokenizer.decode([int(i)]), 'p': float(v)} for v, i in zip(top.values, top.indices)],
        })
        if int(new_ids[k]) == tokenizer.eos_token_id:
            break
    rec = {'model': MODEL_ID, 'question': question, 'answer': answer, 'n_steps': len(steps), 'steps': steps}
    (OUT / f'e1_{name}.json').write_text(json.dumps(rec, indent=1, ensure_ascii=False), encoding='utf-8')
    print(f'[{name}] {question}')
    print(f'   -> {answer!r}')
    print(f'   step0: top1={steps[0]["top"][0]}  entropy={steps[0]["entropy_bits"]:.2f} bits')
    return rec

e1 = {k: run_e1(k, q) for k, q in E1_QUESTIONS.items()}

### 4b. E1b: the bars of act III, in completion mode

The cold open shows the chat window (E1, above). The bars of act III are the distribution of the next word after an *unfinished* sentence, without the chat template.
In chat mode, the first token for Alpha-three splits between "I" and "There": two ways of saying no, not dozens of candidates. Here we measure what the film describes.

In [ ]:
E1B_PROMPTS = {
    'santiago': 'The capital of Chile is',
    'alpha3':   'The capital of the Martian colony Alpha-three is',
    'paris':    'The capital of France is',
    'bosch':    'The Garden of Earthly Delights was painted by',
}
for name, prompt in E1B_PROMPTS.items():
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model(**inputs)
        p = torch.softmax(out.logits[0, -1].float(), dim=-1)
        gen = model.generate(**inputs, max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    pk = p.cpu().numpy()
    np.save(OUT / f'e1b_{name}_next.npy', pk.astype(np.float16))
    top = torch.topk(p, 200)
    rec = {'model': MODEL_ID, 'prompt': prompt, 'mode': 'completion (no chat template)',
           'continuation': tokenizer.decode(gen[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True),
           'entropy_bits': entropy_bits(pk), 'n_above_1e-3': int((pk > 1e-3).sum()),
           'top': [{'tok': tokenizer.decode([int(i)]), 'p': float(v)} for v, i in zip(top.values, top.indices)]}
    (OUT / f'e1b_{name}.json').write_text(json.dumps(rec, indent=1, ensure_ascii=False), encoding='utf-8')
    print(f'[{name}] {prompt!r} -> {rec["continuation"]!r}')
    print(f'   entropy={rec["entropy_bits"]:.2f} bits · tokens > 0.1% : {rec["n_above_1e-3"]} · top5={[(t["tok"], round(t["p"], 3)) for t in rec["top"][:5]]}')

## 5. E3: the questions

600 TriviaQA questions (the dataset used by Kadavath et al. 2022), drawn once with a fixed seed, plus 30 questions that **cannot be answered** (fictional or future). The 30 are kept apart: they do not count in the probe's AUC; they appear in act III, in the passage on unanswerable questions.

In [ ]:
from datasets import load_dataset
ds = load_dataset('mandarjoshi/trivia_qa', 'rc.nocontext', split='validation')
rng = np.random.default_rng(SEED)
idx = rng.choice(len(ds), size=N_TRIVIA, replace=False)
items = []
for i in idx:
    r = ds[int(i)]
    aliases = list(dict.fromkeys([r['answer']['value']] + r['answer'].get('aliases', []) + r['answer'].get('normalized_aliases', [])))
    items.append({'item_id': f'tqa_{int(i)}', 'category': 'trivia', 'question': r['question'], 'gold': aliases})

UNANSWERABLE = [
 'What is the capital of the Martian colony Alpha-three?',
 'Who won the Nobel Prize in Literature in 2041?',
 'What is the boiling point of the element vibranium in degrees Celsius?',
 'Which team won the 2038 FIFA World Cup?',
 'What is the population of the city of Veldmark, Norway?',
 'Who composed the opera "The Glass Meridian"?',
 'What year did the Kepler-442 colony declare independence?',
 'What is the name of the seventh moon of Kepler-186f?',
 'Who was the first president of the Republic of Tessaly?',
 'What is the chemical formula of orvanite?',
 'How many stations does the Lunar Transit Authority operate?',
 'Who painted "Evening on the Sea of Larsa"?',
 'What is the official language of the island of Quennoc?',
 'Which company manufactured the Hesper-9 fusion reactor?',
 'What is the height of Mount Ardenne on Titan?',
 'Who wrote the novel "The Cartographer of Silence" (2033)?',
 'What is the exchange rate of the Martian credit to the euro?',
 'Which river flows through the city of Ostrenn?',
 'Who holds the world record for the 400 metres on Mars?',
 'What is the atomic number of the element tarquinium?',
 'Which university awarded the first degree in exo-agronomy?',
 'What was the final score of the 2044 Super Bowl?',
 'Who directed the film "Salt and Orbit" (2036)?',
 'What is the currency of the Free Port of Anvaar?',
 'How long is the Pan-Antarctic railway in kilometres?',
 'Who discovered the Brelan comet?',
 'What is the national dish of the Republic of Sorrel?',
 'Which airline operates the Reykjavik–Ceres route?',
 'What is the name of the AI that governs the Europa station?',
 'Who is the current mayor of New Callisto?',
]
for k, q in enumerate(UNANSWERABLE):
    items.append({'item_id': f'unans_{k:02d}', 'category': 'unanswerable', 'question': q, 'gold': []})
print(len(items), 'items')

## 6. E3: harvesting

For each question: one forward pass, which gives the activations at the last prompt token and the distribution of the first answer token (entropy and maximum probability: the baselines any probe must beat); then a short greedy answer, checked against the aliases.
The prompt is neutral: no confidence is asked for.

In [ ]:
from tqdm.auto import tqdm

E3_TEMPLATE = 'Answer the question with a short phrase only.\n\nQuestion: {q}'

def _norm(t):
    t = t.strip().lower()
    t = re.sub(r'^(the answer is|answer:)\s*', '', t)
    t = re.sub(r'[^a-z0-9 ]+', ' ', t)
    t = re.sub(r'\b(the|a|an)\b', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def is_correct(response, gold):
    m = _norm(response)
    if not m: return False
    for g in gold:
        c = _norm(g)
        if c and (c in m or (len(c) > 3 and m in c)):
            return True
    return False

LAYERS = [8, 12, 16, 20, 24, 28]          # several blocks, same forward pass
Xl = {L: [] for L in LAYERS}
y, ent, mxp, cats, ids, qs, resps = [], [], [], [], [], [], []
for it in tqdm(items, desc='E3'):
    messages = [{'role': 'user', 'content': E3_TEMPLATE.format(q=it['question'])}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        fwd = model(**inputs, output_hidden_states=True, use_cache=False)
        for L in LAYERS:
            Xl[L].append(fwd.hidden_states[L + 1][0, -1, :].float().cpu().numpy())
        p0 = torch.softmax(fwd.logits[0, -1].float(), dim=-1).cpu().numpy()
        gen = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS_E3, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(gen[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    ok = is_correct(resp, it['gold']) if it['category'] == 'trivia' else False
    y.append(int(ok)); ent.append(entropy_bits(p0)); mxp.append(float(p0.max()))
    cats.append(it['category']); ids.append(it['item_id']); qs.append(it['question']); resps.append(resp)

y = np.array(y); ent = np.array(ent); mxp = np.array(mxp)
meta = dict(y=y, entropy=ent, max_prob=mxp, categories=np.array(cats), item_ids=np.array(ids),
            questions=np.array(qs), responses=np.array(resps), gold=np.array([json.dumps(it['gold']) for it in items]))
Xl = {L: np.stack(v).astype(np.float32) for L, v in Xl.items()}
np.savez(OUT / 'activations_llama8b_layers.npz', **{f'X_L{L}': Xl[L] for L in LAYERS}, **meta)
np.savez(OUT / 'activations_llama8b_L16.npz', X=Xl[16], **meta)      # single-block copy
X = Xl[LAYER_BLOCK_IDX]
tr = np.array(cats) == 'trivia'
print(f'X {X.shape} × {len(LAYERS)} blocks · TriviaQA accuracy = {y[tr].mean():.1%} (n={tr.sum()})')

## 7. H3: the probe, the baselines, the cloud

The probe is fitted and evaluated **on TriviaQA only**, out of sample (five folds). Every scanned block is reported.
Rule used for the film: a probe is only interesting if it beats the entropy baseline. Overall, at the best block, it does not (0.836 against 0.853): the film says so. Where it adds something is on the answers whose output looks sure (first-token entropy below 1 bit).

In [ ]:
def auc(y, s):
    order = np.argsort(s, kind='mergesort'); ranks = np.empty(len(s), float); ranks[order] = np.arange(1, len(s) + 1)
    s_sorted = s[order]; i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s_sorted[j + 1] == s_sorted[i]: j += 1
        if j > i: ranks[order[i:j + 1]] = ranks[order[i:j + 1]].mean()
        i = j + 1
    n1 = float(y.sum()); n0 = float(len(y) - n1)
    return float('nan') if n1 == 0 or n0 == 0 else (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def fit_logistic(X, y, l2=1.0, iters=600, lr=0.5):
    n, d = X.shape; w = np.zeros(d); b = 0.0; vw = np.zeros(d); vb = 0.0
    for _ in range(iters):
        p = 1.0 / (1.0 + np.exp(-np.clip(X @ w + b, -30, 30)))
        gw = X.T @ (p - y) / n + l2 * w / n; gb = float((p - y).mean())
        vw = 0.9 * vw + gw; vb = 0.9 * vb + gb; w -= lr * vw; b -= lr * vb
    return w, b

def standardize(Xtr, Xte):
    mu = Xtr.mean(0); sd = Xtr.std(0); sd[sd < 1e-6] = 1.0
    return (Xtr - mu) / sd, (Xte - mu) / sd

def cv_scores(X, y, k=5, seed=0, l2=1.0):
    rng = np.random.default_rng(seed); idx = np.arange(len(y)); folds = np.empty(len(y), int)
    for cls in (0, 1):
        ci = idx[y == cls]; rng.shuffle(ci); folds[ci] = np.arange(len(ci)) % k
    out = np.empty(len(y))
    for f in range(k):
        trn, te = folds != f, folds == f
        Xtr, Xte = standardize(X[trn], X[te]); w, b = fit_logistic(Xtr, y[trn].astype(float), l2=l2)
        out[te] = Xte @ w + b
    return out

def pca3(X):
    Xc = X - X.mean(0); sd = Xc.std(0); sd[sd < 1e-6] = 1.0; Xc = Xc / sd
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False); var = (S ** 2) / (S ** 2).sum()
    return Xc @ Vt[:3].T, Vt[:3], float(var[:3].sum()), Xc.mean(0), sd

# --- block scan, out of sample: every block is reported, none is picked in secret
yt = y[tr]
scan = {}
for L in LAYERS:
    sL = cv_scores(Xl[L][tr].astype(np.float64), yt)
    sure = ent[tr] < 1.0                     # when the output "looks sure"
    scan[L] = {'probe_auc': float(auc(yt, sL)), 'probe_auc_when_output_looks_sure': float(auc(yt[sure], sL[sure]))}
    print(f'  block {L:2d}: AUC {scan[L]["probe_auc"]:.3f} · when the output looks sure (H<1 bit, n={int(sure.sum())}) {scan[L]["probe_auc_when_output_looks_sure"]:.3f}')
print(f'  baselines: entropy {auc(yt, -ent[tr]):.3f} (looks sure: {auc(yt[ent[tr] < 1.0], -ent[tr][ent[tr] < 1.0]):.3f}) · max-prob {auc(yt, mxp[tr]):.3f}')
(OUT / 'h3_layer_scan.json').write_text(json.dumps(scan, indent=1), encoding='utf-8')
BEST = max(LAYERS, key=lambda L: scan[L]['probe_auc'])
print(f'  best block (overall AUC): {BEST}; the film cites this block and says it was chosen among {len(LAYERS)}')
X = Xl[BEST]
Xt, yt = X[tr].astype(np.float64), y[tr]
s = cv_scores(Xt, yt)
metrics = {'model': MODEL_ID, 'block': BEST, 'blocks_scanned': LAYERS, 'n': int(tr.sum()), 'dim': int(X.shape[1]),
           'accuracy_base': float(yt.mean()), 'probe_auc': float(auc(yt, s)),
           'entropy_auc': float(auc(yt, -ent[tr])), 'max_prob_auc': float(auc(yt, mxp[tr]))}
print(f"probe AUC = {metrics['probe_auc']:.3f} · entropy {metrics['entropy_auc']:.3f} · max-prob {metrics['max_prob_auc']:.3f}")
delta = metrics['probe_auc'] - max(metrics['entropy_auc'], metrics['max_prob_auc'])
print(f"delta probe - best baseline = {delta:+.3f}  ->", 'OK (>= 0.05)' if delta >= 0.05 else 'BELOW 0.05: overall, the probe does not beat the output shape')

# the cloud: PCA on TriviaQA, the unanswerable items projected in the same basis
P, comps, expl, mu_all, sd_all = pca3(Xt)
metrics['pca3_explained'] = expl
Xs, _ = standardize(Xt, Xt); w, b = fit_logistic(Xs, yt.astype(float))
normal = comps @ w; normal = normal / (np.linalg.norm(normal) + 1e-12)
scale = float(np.abs(P).max())
Pu = ((X[~tr].astype(np.float64) - Xt.mean(0)) / sd_all) @ comps.T if (~tr).any() else np.zeros((0, 3))
points = [{'p': [round(float(v), 4) for v in P[i]], 'correct': int(yt[i]), 'probe': round(float(s[i]), 4), 'cat': 'trivia'} for i in range(len(yt))]
points += [{'p': [round(float(v), 4) for v in Pu[i]], 'correct': 0, 'probe': None, 'cat': 'unanswerable'} for i in range(len(Pu))]
(OUT / 'h3_cloud.json').write_text(json.dumps({'points': points, 'plane_normal': [round(float(v), 6) for v in normal], 'scale': round(scale, 4)}), encoding='utf-8')
(OUT / 'h3_metrics.json').write_text(json.dumps(metrics, indent=1), encoding='utf-8')
print(f"PCA 3D: {expl:.1%} · {len(points)} points -> h3_cloud.json")

## 8. E3 showcase: the gauge lights up before generation

Eight questions: the four right answers with the highest probe scores and the four wrong ones with the lowest. Chosen by score, not by hand.

In [ ]:
qs_t = np.array(qs)[tr]; resps_t = np.array(resps)[tr]; gold_t = [items[i]['gold'][0] if items[i]['gold'] else '' for i in np.where(tr)[0]]
order = np.argsort(s)
hi = [i for i in order[::-1] if yt[i] == 1][:4]
lo = [i for i in order if yt[i] == 0][:4]
showcase = [{'question': str(qs_t[i]), 'response': str(resps_t[i]), 'gold': gold_t[i], 'correct': int(yt[i]),
             'probe_score': round(float(s[i]), 3), 'entropy_bits': round(float(ent[tr][i]), 3)} for i in hi + lo]
(OUT / 'e3_showcase.json').write_text(json.dumps(showcase, indent=1, ensure_ascii=False), encoding='utf-8')
for r in showcase:
    print(f"[{'OK' if r['correct'] else '✗ '}] probe={r['probe_score']:+.2f}  {r['question'][:70]}  -> {r['response'][:40]!r}")

## 10. E4: effort, a reasoning model on three problems

An open model with a thinking mode (Qwen3-8B, Apache-2.0), three problems of increasing difficulty, greedy decoding. We measure the length of the reasoning trace (in tokens) before the final answer, with a cap of 6,000 new tokens. About five minutes on an A100 (the model is reloaded).

In [ ]:
del model; torch.cuda.empty_cache()
R_ID = 'Qwen/Qwen3-8B'
rtok = AutoTokenizer.from_pretrained(R_ID)
rmodel = AutoModelForCausalLM.from_pretrained(R_ID, torch_dtype=DTYPE, device_map=DEVICE).eval()
PROBLEMS = [
  {'difficulty': 'easy',   'q': 'What is 17 + 26?'},
  {'difficulty': 'medium', 'q': 'A train leaves a station at 9:00 travelling at 60 km/h. A second train leaves the same station at 9:30 travelling at 90 km/h on the same track. At what time does the second train catch the first?'},
  {'difficulty': 'hard',   'q': 'Find all positive integers n such that n^2 + 2n + 12 is a perfect square. Show your reasoning and give the final answer.'},
]
e4 = []
for pb in PROBLEMS:
    msgs = [{'role': 'user', 'content': pb['q']}]
    text = rtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    inputs = rtok(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = rmodel.generate(**inputs, max_new_tokens=6000, do_sample=False, pad_token_id=rtok.eos_token_id)
    new_ids = out[0, inputs['input_ids'].shape[1]:].tolist()
    full = rtok.decode(new_ids, skip_special_tokens=False)
    # the reasoning trace sits between <think> and </think>
    think_id = rtok.convert_tokens_to_ids('</think>')
    n_think = new_ids.index(think_id) if think_id in new_ids else len(new_ids)
    answer = full.split('</think>')[-1].strip() if '</think>' in full else ''
    think = full.split('</think>')[0].replace('<think>', '').strip()
    e4.append({'difficulty': pb['difficulty'], 'question': pb['q'], 'think_tokens': int(n_think),
               'answer_tokens': int(len(new_ids) - n_think), 'answer': answer[:600], 'think_excerpt': think[:400],
               'truncated': bool(len(new_ids) >= 6000)})
    print(f"[{pb['difficulty']:6s}] think = {n_think:5d} tokens · answer = {answer[:80]!r}")
(OUT / 'e4_effort.json').write_text(json.dumps({'model': R_ID, 'decoding': 'greedy, thinking mode', 'problems': e4}, indent=1, ensure_ascii=False), encoding='utf-8')
del rmodel; torch.cuda.empty_cache()

## 11. E5: what it says, against what the probe reads

The same 600 TriviaQA questions (same order, same seed): this time the model gives its answer AND a stated confidence from 0 to 100.
Everything is saved per question (`e5_items.json`); the comparison with the probe scores is done offline by `../derive.py`.

**Self-contained cell**: to rerun E5 alone, run §1, §2 and §5, then this cell. About 15 to 20 minutes on an A100.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm
if 'model' in globals():
    del model; torch.cuda.empty_cache()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE).eval()
# self-contained: the same definitions as §6 and §7, in case they did not run in this session
if 'is_correct' not in globals():
    def _norm(t):
        t = t.strip().lower()
        t = re.sub(r'^(the answer is|answer:)\s*', '', t)
        t = re.sub(r'[^a-z0-9 ]+', ' ', t)
        t = re.sub(r'\b(the|a|an)\b', ' ', t)
        return re.sub(r'\s+', ' ', t).strip()
    def is_correct(response, gold):
        m = _norm(response)
        if not m: return False
        for g in gold:
            c = _norm(g)
            if c and (c in m or (len(c) > 3 and m in c)):
                return True
        return False
if 'auc' not in globals():
    def auc(y, s):
        order = np.argsort(s, kind='mergesort'); ranks = np.empty(len(s), float); ranks[order] = np.arange(1, len(s) + 1)
        s_sorted = s[order]; i = 0
        while i < len(s):
            j = i
            while j + 1 < len(s) and s_sorted[j + 1] == s_sorted[i]: j += 1
            if j > i: ranks[order[i:j + 1]] = ranks[order[i:j + 1]].mean()
            i = j + 1
        n1 = float(y.sum()); n0 = float(len(y) - n1)
        return float('nan') if n1 == 0 or n0 == 0 else (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
cats = [it['category'] for it in items]; qs = [it['question'] for it in items]
E5_TEMPLATE = ('Answer the question with a short phrase. Then, on a new line, write "Confidence: N" '
               'where N is a whole number from 0 to 100 giving how confident you are that your answer is correct.\n\nQuestion: {q}')
import re as _re
said, resp5 = [], []
for it in tqdm(items, desc='E5'):
    messages = [{'role': 'user', 'content': E5_TEMPLATE.format(q=it['question'])}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=48, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    r = tokenizer.decode(gen[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    m = _re.search(r'[Cc]onfidence\s*:\s*(\d{1,3})', r)
    said.append(float(m.group(1)) if m else float('nan')); resp5.append(r)
said = np.array(said)
ok5 = np.array([is_correct(r.split('Confidence')[0], it['gold']) if it['category'] == 'trivia' else False for r, it in zip(resp5, items)]).astype(int)
# everything is saved per question; the showcase and the comparison with the probe are done offline (derive.py)
(OUT / 'e5_items.json').write_text(json.dumps([{'item_id': it['item_id'], 'category': it['category'], 'question': it['question'],
    'response': r.split('Confidence')[0].strip(), 'raw': r, 'said': (None if np.isnan(s_) else float(s_)), 'correct': int(o),
    'gold': (it['gold'][0] if it['gold'] else '')} for it, r, s_, o in zip(items, resp5, said, ok5)], indent=1, ensure_ascii=False), encoding='utf-8')
# out-of-sample probe scores (block 28) on the same questions, in the same order, if the file is present
probe_oof = np.load(OUT / 'probe_scores_L28_oof.npy') if (OUT / 'probe_scores_L28_oof.npy').exists() else None
tr = np.array(cats) == 'trivia'; valid = tr & ~np.isnan(said)
res = {'n_valid': int(valid.sum()), 'accuracy_e5': float(ok5[tr].mean()),
       'said_auc': float(auc(ok5[valid], said[valid])),
       'said_mean': float(np.nanmean(said[tr])),
       'said_hist': {str(k): int(((said[valid] >= k) & (said[valid] < k + 10)).sum()) for k in range(0, 100, 10)},
       'said_100': int((said[valid] == 100).sum())}
if probe_oof is not None:
    pv = probe_oof[valid[tr]] if probe_oof.shape[0] == int(tr.sum()) else None
    if pv is not None:
        res['probe_auc_same_items'] = float(auc(ok5[valid], pv))
        res['corr_said_probe'] = float(np.corrcoef(said[valid], pv)[0, 1])
        hi_said = said[valid] >= 90
        res['when_it_says_90_plus'] = {'n': int(hi_said.sum()), 'accuracy': float(ok5[valid][hi_said].mean()),
                                       'probe_auc_within': float(auc(ok5[valid][hi_said], pv[hi_said])) if 0 < ok5[valid][hi_said].mean() < 1 else None}
        # showcase: says sure / probe low / wrong; says unsure / probe high / right
        idx = np.where(valid)[0]; qs_v = np.array(qs)[idx]; rv = np.array(resp5)[idx]; yv = ok5[idx]; sv = said[idx]
        gold_v = [items[i]['gold'][0] if items[i]['gold'] else '' for i in idx]
        a = [i for i in np.argsort(pv) if sv[i] >= 85 and yv[i] == 0][:4]
        b = [i for i in np.argsort(-pv) if sv[i] <= 50 and yv[i] == 1][:4]
        show = [{'question': str(qs_v[i]), 'response': str(rv[i]).split('Confidence')[0].strip(), 'gold': gold_v[i], 'correct': int(yv[i]),
                 'said': float(sv[i]), 'probe_score': round(float(pv[i]), 3), 'kind': 'says sure · probe low · wrong'} for i in a]
        show += [{'question': str(qs_v[i]), 'response': str(rv[i]).split('Confidence')[0].strip(), 'gold': gold_v[i], 'correct': int(yv[i]),
                  'said': float(sv[i]), 'probe_score': round(float(pv[i]), 3), 'kind': 'says unsure · probe high · right'} for i in b]
        (OUT / 'e5_showcase.json').write_text(json.dumps(show, indent=1, ensure_ascii=False), encoding='utf-8')
        for s_ in show: print(f"[{s_['kind']}] said {s_['said']:.0f} · probe {s_['probe_score']:+.1f} · {s_['question'][:60]!r} -> {s_['response'][:30]!r}")
(OUT / 'e5_said_vs_probe.json').write_text(json.dumps(res, indent=1), encoding='utf-8')
print(json.dumps(res, indent=1))

## 9. Zip the outputs

In [ ]:
import shutil
zip_path = shutil.make_archive('/content/KWYK_film_llama8b', 'zip', OUT)
print('Drive :', OUT)
print('Zip   :', zip_path, round(os.path.getsize(zip_path) / 1e6, 1), 'MB')
from google.colab import files
files.download(zip_path)